In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ball = pd.read_parquet(
    "../data/processed/primary_ball.parquet"
)

matches = pd.read_csv(
    "../data/processed/primary_matches.csv"
)

print("Data loaded successfully!")

Data loaded successfully!


In [34]:

# Inspect original match metadata

print("Original match metadata:")

print(
    matches[
        [
            "match_id",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target"
        ]
    ].drop_duplicates().head(20).to_string(index=False)
)

print("\nUnique first innings values:")
print(
    sorted(matches["first_inn_revised_overs"].dropna().unique())
)

print("\nUnique second innings values:")
print(
    sorted(matches["second_inn_revised_overs"].dropna().unique())
)


Original match metadata:
 match_id  first_inn_revised_overs  second_inn_revised_overs  revised_target
    14094                      NaN                       NaN             NaN
    14095                      NaN                       NaN             NaN
    14096                      NaN                       NaN             NaN
    14097                      NaN                       NaN             NaN
    14098                      NaN                       NaN             NaN
    14099                      NaN                       NaN             NaN
    14100                      NaN                       NaN             NaN
    14101                      NaN                       NaN             NaN
    14102                      NaN                       NaN             NaN
    14103                      NaN                       NaN             NaN
    14104                      NaN                       NaN             NaN
    14105                      NaN                 

In [35]:
ball = ball.sort_values(
    ["match_id", "innings", "innings_ball_num"]
).copy()

# Total runs scored on each delivery
ball["total_runs"] = (
    ball["runs"]
    + ball["extras1_runs"]
    + ball["extras_2_runs"]
)

# Illegal deliveries
ball["is_wide"] = ball["extras_type"].eq("wide")
ball["is_no_ball"] = ball["extras_type"].eq("Overstep")

ball["legal_ball"] = ~(
    ball["is_wide"] | ball["is_no_ball"]
)

# Wickets lost by batting team
ball["wicket"] = ball["is_wicket"].astype(int)

print("Delivery preparation complete!")

Delivery preparation complete!


In [36]:
group = ball.groupby(
    ["match_id", "innings"],
    sort=False
)

# Score before current delivery
ball["score_before"] = group["total_runs"].transform(
    lambda x: x.cumsum().shift(1, fill_value=0)
)

# Wickets lost before current delivery
ball["wickets_before"] = group["wicket"].transform(
    lambda x: x.cumsum().shift(1, fill_value=0)
)

# Legal balls bowled before current delivery
ball["legal_balls_before"] = group["legal_ball"].transform(
    lambda x: x.astype(int).cumsum().shift(1, fill_value=0)
)

ball["wickets_remaining"] = (
    10 - ball["wickets_before"]
)

print(
    ball[
        ["match_id", "innings", "striker_name",
         "score_before", "wickets_before",
         "legal_balls_before", "wickets_remaining"]
    ].head(10)
)

   match_id  innings striker_name  score_before  wickets_before  \
0     13511        1    Jason Roy             0               0   
1     13511        1    Jason Roy             0               0   
2     13511        1    Jason Roy             0               0   
3     13511        1    Jason Roy             0               0   
4     13511        1    Jason Roy             0               0   
5     13511        1    Jason Roy             6               0   
6     13511        1   Tom Banton             6               1   
7     13511        1   Tom Banton             6               1   
8     13511        1   Tom Banton             6               1   
9     13511        1   Tom Banton             6               1   

   legal_balls_before  wickets_remaining  
0                   0                 10  
1                   1                 10  
2                   2                 10  
3                   3                 10  
4                   4                 10  
5   

In [37]:
print("Available columns:")
print(ball.columns.tolist())

print("\nRequired columns:")
required = [
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target",
    "legal_balls_before"
]

for col in required:
    print(col, ":", col in ball.columns)

Available columns:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time', 'comp_id', 'comp_name', 'match_format', 'season', 'match_date', 'venue_country', 'revised_target', 'first_inn_revised_overs', 'second_inn_revised_overs', 'total_runs', 'is_wide', 'is_no_ball', 'legal_ball', 'wicket', 'score_before', 'wickets_before', 'legal_balls_before', 'wickets_remaining']

Required columns:
first_inn_revised_overs : True
second_inn_revised_overs : True
revised_target : True
legal_balls_before : True


In [38]:

# STEP 12: Reconstruct innings duration

# Prepare match metadata
revision_data = matches[
    [
        "match_id",
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target"
    ]
].drop_duplicates(subset="match_id").rename(
    columns={
        "first_inn_revised_overs": "first_inn_limit",
        "second_inn_revised_overs": "second_inn_limit",
        "revised_target": "match_revised_target"
    }
)

# Remove old metadata columns before merging
old_cols = [
    "first_inn_limit",
    "second_inn_limit",
    "match_revised_target",
    "innings_limit",
    "innings_overs",
    "max_legal_balls",
    "balls_remaining"
]

ball = ball.drop(
    columns=[c for c in old_cols if c in ball.columns],
    errors="ignore"
)

# Merge match information
ball = ball.merge(
    revision_data,
    on="match_id",
    how="left",
    validate="many_to_one"
)

# Select innings-specific legal-ball limit
ball["innings_limit"] = np.where(
    ball["innings"] == 1,
    ball["first_inn_limit"],
    ball["second_inn_limit"]
)

# Standard T20 duration = 120 legal balls
ball["max_legal_balls"] = (
    ball["innings_limit"].fillna(120)
)

ball["max_legal_balls"] = (
    ball["max_legal_balls"].round().astype(int)
)

# Convert legal balls to overs for display
ball["innings_overs"] = (
    ball["max_legal_balls"] / 6
)

# Calculate remaining legal balls
ball["balls_remaining"] = (
    ball["max_legal_balls"]
    - ball["legal_balls_before"]
).clip(lower=0)

print("STEP 12 COMPLETED!")

print(
    ball[
        [
            "match_id",
            "innings",
            "innings_overs",
            "max_legal_balls",
            "legal_balls_before",
            "balls_remaining"
        ]
    ].head(10)
)


STEP 12 COMPLETED!
   match_id  innings  innings_overs  max_legal_balls  legal_balls_before  \
0     13511        1           20.0              120                   0   
1     13511        1           20.0              120                   1   
2     13511        1           20.0              120                   2   
3     13511        1           20.0              120                   3   
4     13511        1           20.0              120                   4   
5     13511        1           20.0              120                   5   
6     13511        1           20.0              120                   6   
7     13511        1           20.0              120                   7   
8     13511        1           20.0              120                   8   
9     13511        1           20.0              120                   9   

   balls_remaining  
0              120  
1              119  
2              118  
3              117  
4              116  
5              115

In [39]:
print("Missing innings limits:",
      ball["max_legal_balls"].isna().sum())

print("Negative balls remaining:",
      (ball["balls_remaining"] < 0).sum())

print("\nInnings duration distribution:")

print(
    ball["max_legal_balls"]
    .value_counts()
    .sort_index()
)

print("\nRevised target matches:")

print(
    revision_data["match_revised_target"]
    .notna()
    .sum()
)

Missing innings limits: 0
Negative balls remaining: 0

Innings duration distribution:
max_legal_balls
30         97
42        126
48        187
54        173
60        184
66        288
72        100
78        141
84        267
90        610
102       420
108       173
114       504
120    190744
Name: count, dtype: int64

Revised target matches:
23


In [40]:
shortened = ball[
    ball["max_legal_balls"] < 120
]

print("Shortened innings records:")

print(
    shortened[
        [
            "match_id",
            "innings",
            "innings_overs",
            "max_legal_balls",
            "legal_balls_before",
            "balls_remaining"
        ]
    ].drop_duplicates(
        ["match_id", "innings"]
    ).head(20)
)

Shortened innings records:
        match_id  innings  innings_overs  max_legal_balls  legal_balls_before  \
2114       13612        2           19.0              114                   0   
22050      15410        1           19.0              114                   0   
22756      15546        1           14.0               84                   0   
34099      16753        1           11.0               66                   0   
34331      16754        2            7.0               42                   0   
35996      17395        1            8.0               48                   0   
36045      17395        2            8.0               48                   0   
41945      17765        1           12.0               72                   0   
42020      17765        2           12.0               72                   0   
44622      19959        1           11.0               66                   0   
63751      21070        2            8.0               48                   0   
6

In [41]:

# STEP 13: Calculate first innings totals and chase targets

# Remove columns from previous executions
old_cols = [
    "first_innings_total",
    "target",
    "runs_required",
    "required_run_rate"
]

ball = ball.drop(
    columns=[c for c in old_cols if c in ball.columns],
    errors="ignore"
)

# Calculate first innings total for every match
first_innings = (
    ball[ball["innings"] == 1]
    .groupby("match_id")["total_runs"]
    .sum()
)

# Map first innings totals without another merge
ball["first_innings_total"] = (
    ball["match_id"].map(first_innings)
)

# Use revised target when available
ball["target"] = ball["match_revised_target"].where(
    ball["match_revised_target"].notna(),
    ball["first_innings_total"] + 1
)

# Calculate runs required before each delivery
ball["runs_required"] = np.where(
    ball["innings"] == 2,
    ball["target"] - ball["score_before"],
    np.nan
)

# Calculate required run rate
ball["required_run_rate"] = np.where(
    (ball["innings"] == 2)
    & (ball["balls_remaining"] > 0),
    ball["runs_required"] * 6
    / ball["balls_remaining"],
    np.nan
)

print("STEP 13 COMPLETED SUCCESSFULLY!")

print(
    ball[
        [
            "match_id",
            "innings",
            "score_before",
            "first_innings_total",
            "target",
            "runs_required",
            "balls_remaining",
            "required_run_rate"
        ]
    ].tail(20)
)


STEP 13 COMPLETED SUCCESSFULLY!
        match_id  innings  score_before  first_innings_total  target  \
193994     55561        2            44                   79    80.0   
193995     55561        2            46                   79    80.0   
193996     55561        2            48                   79    80.0   
193997     55561        2            48                   79    80.0   
193998     55561        2            54                   79    80.0   
193999     55561        2            54                   79    80.0   
194000     55561        2            54                   79    80.0   
194001     55561        2            54                   79    80.0   
194002     55561        2            58                   79    80.0   
194003     55561        2            62                   79    80.0   
194004     55561        2            63                   79    80.0   
194005     55561        2            64                   79    80.0   
194006     55561        2       

In [42]:
print("MATCH STATE VALIDATION")

checks = {
    "Missing score":
        ball["score_before"].isna().sum(),

    "Negative wickets remaining":
        (ball["wickets_remaining"] < 0).sum(),

    "Negative balls remaining":
        (ball["balls_remaining"] < 0).sum(),

    "Missing innings duration":
        ball["max_legal_balls"].isna().sum(),

    "Missing total runs":
        ball["total_runs"].isna().sum(),

    "Missing first innings total in second innings":
        ball.loc[
            ball["innings"] == 2,
            "first_innings_total"
        ].isna().sum(),

    "Missing target in second innings":
        ball.loc[
            ball["innings"] == 2,
            "target"
        ].isna().sum()
}

for name, value in checks.items():
    print(f"{name}: {value}")

MATCH STATE VALIDATION
Missing score: 0
Negative wickets remaining: 0
Negative balls remaining: 0
Missing innings duration: 0
Missing total runs: 0
Missing first innings total in second innings: 0
Missing target in second innings: 0


In [43]:
print(
    ball[ball["match_revised_target"].notna()][
        [
            "match_id",
            "innings",
            "first_innings_total",
            "match_revised_target",
            "target",
            "score_before",
            "runs_required"
        ]
    ].drop_duplicates(
        ["match_id", "innings"]
    ).to_string(index=False)
)

 match_id  innings  first_innings_total  match_revised_target  target  score_before  runs_required
    13612        1                  149                 143.0   143.0             0            NaN
    13612        2                  149                 143.0   143.0             0          143.0
    16754        1                   95                  56.0    56.0             0            NaN
    16754        2                   95                  56.0    56.0             0           56.0
    17395        1                   90                  91.0    91.0             0            NaN
    17395        2                   90                  91.0    91.0             0           91.0
    17765        1                  112                 130.0   130.0             0            NaN
    17765        2                  112                 130.0   130.0             0          130.0
    21070        1                  207                 104.0   104.0             0            NaN
    21070 

In [44]:
revised_matches = matches[
    matches["revised_target"].notna()
][
    [
        "match_id",
        "match_date",
        "comp_name",
        "match_format",
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target"
    ]
].drop_duplicates("match_id")

print(revised_matches.to_string(index=False))

 match_id match_date                                    comp_name match_format  first_inn_revised_overs  second_inn_revised_overs  revised_target
    21586 2023-05-29                                     IPL 2023          T20                      NaN                      90.0           171.0
    40727 2025-04-18                                     IPL 2025          T20                     84.0                      84.0            96.0
    40749 2025-05-06                                     IPL 2025          T20                      NaN                     114.0           147.0
    50137 2026-04-07                                     IPL 2026          T20                     66.0                      66.0           151.0
    50371 2026-05-07                                     IPL 2026          T20                    114.0                     114.0           213.0
    13612 2022-02-11      Sri Lanka in Australia T20I Series 2022         T20I                      NaN                     

In [45]:
revised_ids = revised_matches["match_id"]

check = ball[
    (ball["match_id"].isin(revised_ids))
    & (ball["innings"] == 2)
].groupby("match_id").agg(
    first_innings_total=("first_innings_total", "first"),
    revised_target=("match_revised_target", "first"),
    second_innings_runs=("total_runs", "sum"),
    second_innings_balls=("legal_ball", "sum"),
    innings_limit=("max_legal_balls", "first")
).reset_index()

check["second_innings_runs_needed"] = (
    check["revised_target"] - check["second_innings_runs"]
)

print(check.to_string(index=False))

 match_id  first_innings_total  revised_target  second_innings_runs  second_innings_balls  innings_limit  second_innings_runs_needed
    13612                  149           143.0                  122                   114            114                        21.0
    16754                   95            56.0                   56                    40             42                         0.0
    17395                   90            91.0                   92                    44             48                        -1.0
    17765                  112           130.0                   30                    23             72                       100.0
    21070                  207           104.0                   81                    48             48                        23.0
    21071                  202           203.0                  125                   102            102                        78.0
    21586                  214           171.0                  171  

In [46]:
# STEP 14: Check when the chasing team reaches its target

second = ball[ball["innings"] == 2].copy()

second["score_after"] = (
    second["score_before"] + second["total_runs"]
)

second["target_reached"] = (
    second["score_after"] >= second["target"]
)

reached = (
    second[second["target_reached"]]
    .groupby("match_id")
    .first()
    .reset_index()
)

print("Matches where target was reached:")
print(
    reached[
        [
            "match_id",
            "target",
            "score_before",
            "total_runs",
            "score_after",
            "balls_remaining"
        ]
    ].to_string(index=False)
)

print("\nNumber of matches where target was reached:",
      reached["match_id"].nunique())

print("\nTotal second-innings matches:",
      second["match_id"].nunique())

Matches where target was reached:
 match_id  target  score_before  total_runs  score_after  balls_remaining
    13511   104.0           103           1          104               18
    13522   158.0           156           6          162                8
    13614   122.0           120           4          124               19
    13615   140.0           139           4          143               12
    13616   155.0           154           1          155                2
    14094   132.0           129           4          133               10
    14095   178.0           175           4          179               11
    14096   206.0           204           4          208                7
    14097   159.0           157           4          161                3
    14099   129.0           128           4          132                5
    14100   211.0           210           1          211                4
    14101   138.0           135           6          141               34
    

In [47]:
# STEP 15: Inspect extras categories

print("EXTRAS TYPE COUNTS")
print(
    ball["extras_type"]
    .fillna("No extras")
    .value_counts(dropna=False)
)

print("\nEXTRAS RUNS SUMMARY")
print(
    ball.groupby("extras_type", dropna=False)[
        ["extras1_runs", "extras_2_runs"]
    ].sum()
)

EXTRAS TYPE COUNTS
extras_type
No extras    182548
wide           7474
legbyes        2661
Overstep        565
byes            505
other           260
Penalty           1
Name: count, dtype: int64

EXTRAS RUNS SUMMARY
             extras1_runs  extras_2_runs
extras_type                             
Overstep              565             48
Penalty                 5              0
byes                  960              0
legbyes              3500              0
other                 260             34
wide                 8882              0
NaN                     1              1


In [50]:
# STEP 16: Inspect dismissal types

dismissals = ball[ball["is_wicket"] == 1]

print("DISMISSAL TYPE COUNTS")

print(
    dismissals["dismissal_type"]
    .fillna("Unknown")
    .value_counts(dropna=False)
)

print("\nDISMISSAL TYPE IDs")

print(
    dismissals[
        ["dismissal_type_id", "dismissal_type"]
    ].drop_duplicates().sort_values("dismissal_type_id")
)

DISMISSAL TYPE COUNTS
dismissal_type
Caught                   7332
Bowled                   1690
LBW                       666
RunOut                    573
Stump                     200
Hit Wicket                 17
Retire Out                  6
Obstracting the field       1
Obstructing the field       1
Name: count, dtype: int64

DISMISSAL TYPE IDs
        dismissal_type_id         dismissal_type
5                     1.0                 Bowled
10                    2.0                 Caught
354                   3.0                    LBW
63                    4.0                 RunOut
42                    5.0                  Stump
15564                 6.0             Hit Wicket
7903                  9.0             Retire Out
96293                10.0  Obstracting the field
178041               10.0  Obstructing the field


In [51]:
# STEP 17: Inspect unusual extras

unusual = ball[
    ball["extras_type"].isin(["other", "Penalty"])
    | (
        ball["extras_type"].isna()
        & (
            (ball["extras1_runs"] != 0)
            | (ball["extras_2_runs"] != 0)
        )
    )
]

print("Unusual extras records:", len(unusual))

print(
    unusual[
        [
            "match_id",
            "innings",
            "overid",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].to_string(index=False)
)

Unusual extras records: 263
 match_id  innings  overid           bowler_name  runs extras_type  extras1_runs  extras_2_runs  is_wicket dismissal_type
    13513        1      10           Tymal Mills     0       other             1              0      False            NaN
    13515        1      16         Saqib Mahmood     0       other             1              0      False            NaN
    13515        2      19      Sheldon Cottrell     2       other             1              0      False            NaN
    13614        1      18        Mitchell Starc     0       other             1              4      False            NaN
    13614        2       2        Nuwan Thushara     0       other             1              0      False            NaN
    14094        1      19           Shivam Mavi     6       other             1              0      False            NaN
    14098        2      20   Nathan Coulter Nile     0       other             1              0      False            

In [52]:
# STEP 18: Detailed extras investigation

print("OTHER EXTRAS: COMBINATION COUNTS")

print(
    ball[ball["extras_type"] == "other"]
    .groupby(["extras1_runs", "extras_2_runs"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .to_string(index=False)
)

print("\nMISSING EXTRAS TYPE RECORDS")

print(
    ball[ball["extras_type"].isna()][
        [
            "match_id",
            "innings",
            "overid",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].to_string(index=False)
)

print("\nPENALTY RECORDS")

print(
    ball[ball["extras_type"] == "Penalty"][
        [
            "match_id",
            "innings",
            "overid",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "bowler_name"
        ]
    ].to_string(index=False)
)

OTHER EXTRAS: COMBINATION COUNTS
 extras1_runs  extras_2_runs  count
            1              0    247
            1              4      7
            1              1      6

MISSING EXTRAS TYPE RECORDS
 match_id  innings  overid  runs  extras1_runs  extras_2_runs  is_wicket        dismissal_type
    13511        1       1     0             0              0      False                   NaN
    13511        1       1     0             0              0      False                   NaN
    13511        1       1     0             0              0      False                   NaN
    13511        1       1     0             0              0      False                   NaN
    13511        1       1     6             0              0      False                   NaN
    13511        1       1     0             0              0       True                Bowled
    13511        1       2     0             0              0      False                   NaN
    13511        1       2     0  

In [53]:
# STEP 19: Compare batter runs and extras

print("RUNS AND EXTRAS SAMPLE")

print(
    ball[
        ball["extras_type"].notna()
    ][
        [
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].head(25).to_string(index=False)
)

print("\nTOTAL RUNS CHECK")

print(
    ball[
        ["runs", "extras1_runs", "extras_2_runs"]
    ].sum()
)

RUNS AND EXTRAS SAMPLE
 runs extras_type  extras1_runs  extras_2_runs
    0        wide             1              0
    0        wide             1              0
    0        wide             1              0
    0     legbyes             4              0
    0     legbyes             1              0
    0        wide             1              0
    0        wide             1              0
    0        wide             1              0
    0        wide             1              0
    0        wide             1              0
    0        wide             1              0
    0        wide             1              0
    0        wide             1              0
    0        wide             2              0
    0     legbyes             1              0
    0     legbyes             1              0
    0        byes             2              0
    0        byes             1              0
    0        byes             1              0
    0     legbyes             1      

In [54]:
# STEP 20: Compact investigation of other extras

other = ball[ball["extras_type"] == "other"].copy()

print("Other extras count:", len(other))

print("\nUnique matches containing other extras:")
print(other["match_id"].nunique())

print("\nSample records:")
print(
    other[
        [
            "match_id",
            "innings",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "bowler_name"
        ]
    ].head(20).to_string(index=False)
)

print("\nMissing extras type records:")
print(ball["extras_type"].isna().sum())

print("\nMissing extras type with non-zero extras:")
print(
    ball[
        ball["extras_type"].isna()
        & (
            (ball["extras1_runs"] != 0)
            | (ball["extras_2_runs"] != 0)
        )
    ][
        ["match_id", "runs", "extras1_runs", "extras_2_runs"]
    ].to_string(index=False)
)

Other extras count: 260

Unique matches containing other extras:
211

Sample records:
 match_id  innings  runs  extras1_runs  extras_2_runs         bowler_name
    13513        1     0             1              0         Tymal Mills
    13515        1     0             1              0       Saqib Mahmood
    13515        2     2             1              0    Sheldon Cottrell
    13614        1     0             1              4      Mitchell Starc
    13614        2     0             1              0      Nuwan Thushara
    14094        1     6             1              0         Shivam Mavi
    14098        2     0             1              0 Nathan Coulter Nile
    14101        1     0             1              0         Tim Southee
    14102        2     0             1              0     Prasidh Krishna
    14107        2     2             1              0         Daniel Sams
    14108        2     6             1              0       Anrich Nortje
    14108        2     0  

In [55]:
# STEP 21: Check dismissals on extra deliveries

extra_dismissals = ball[
    (ball["is_wicket"] == 1)
    & (ball["extras_type"].notna())
]

print("Dismissals with extras:", len(extra_dismissals))

print(
    extra_dismissals[
        [
            "match_id",
            "innings",
            "extras_type",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "dismissal_type"
        ]
    ].to_string(index=False)
)

Dismissals with extras: 47
 match_id  innings extras_type  runs  extras1_runs  extras_2_runs dismissal_type
    14094        1        wide     0             1              0          Stump
    14102        2       other     0             1              0         RunOut
    14145        1       other     0             1              0         RunOut
    14164        1    Overstep     0             1              0         RunOut
    15187        1    Overstep     1             1              0         RunOut
    15187        1        wide     0             1              0         RunOut
    15609        1        wide     0             1              0          Stump
    15610        1        wide     0             1              0         RunOut
    15611        2        wide     0             1              0          Stump
    16866        1        wide     0             1              0          Stump
    16867        1        wide     0             1              0         RunOut
 

In [56]:
# STEP 22: Investigate missing extras categories

missing_extras = ball[
    ball["extras_type"].isna()
    & (
        (ball["extras1_runs"] != 0)
        | (ball["extras_2_runs"] != 0)
    )
]

print(
    missing_extras[
        [
            "match_id",
            "innings",
            "innings_ball_num",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type",
            "bowler_name"
        ]
    ].to_string(index=False)
)

 match_id  innings  innings_ball_num  runs  extras1_runs  extras_2_runs  is_wicket dismissal_type      bowler_name
    17555        1                29     0             0              1      False            NaN       Haris Rauf
    36097        2                49     0             1              0      False            NaN Romario Shepherd


In [57]:
# STEP 23: Investigate other extras

other = ball[ball["extras_type"] == "other"]

print("Other extras combinations:")

print(
    other.groupby(
        ["runs", "extras1_runs", "extras_2_runs"]
    ).size().reset_index(name="count")
    .sort_values("count", ascending=False)
    .to_string(index=False)
)

print("\nOther extras with dismissals:")

print(
    other[other["is_wicket"] == 1][
        [
            "match_id",
            "innings",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "dismissal_type"
        ]
    ].to_string(index=False)
)

Other extras combinations:
 runs  extras1_runs  extras_2_runs  count
    0             1              0    112
    1             1              0     48
    4             1              0     42
    6             1              0     26
    2             1              0     18
    0             1              4      7
    0             1              1      6
    3             1              0      1

Other extras with dismissals:
 match_id  innings  runs  extras1_runs  extras_2_runs dismissal_type
    14102        2     0             1              0         RunOut
    14145        1     0             1              0         RunOut


In [58]:
# STEP 24: Create validated delivery variables

# 1. Team runs
ball["team_runs"] = (
    ball["runs"].fillna(0)
    + ball["extras1_runs"].fillna(0)
    + ball["extras_2_runs"].fillna(0)
)

# 2. Known illegal deliveries
ball["is_wide"] = ball["extras_type"].eq("wide")
ball["is_no_ball"] = ball["extras_type"].eq("Overstep")

# 3. Flag uncertain extras
ball["uncertain_extras"] = (
    ball["extras_type"].eq("other")
    | ball["extras_type"].eq("Penalty")
    | (
        ball["extras_type"].isna()
        & (
            ball["extras1_runs"].fillna(0).ne(0)
            | ball["extras_2_runs"].fillna(0).ne(0)
        )
    )
)

# 4. Legal delivery flag
ball["legal_ball"] = ~(
    ball["is_wide"] | ball["is_no_ball"]
)

# 5. Team wickets
ball["team_wicket"] = (
    ball["is_wicket"].fillna(False).astype(int)
)

# 6. Bowler-credited wickets
bowler_wicket_types = [
    "Caught",
    "Bowled",
    "LBW",
    "Stump",
    "Hit Wicket"
]

ball["bowler_wicket"] = (
    ball["dismissal_type"]
    .isin(bowler_wicket_types)
    .astype(int)
)

print("Team runs:", ball["team_runs"].sum())
print("Team wickets:", ball["team_wicket"].sum())
print("Bowler wickets:", ball["bowler_wicket"].sum())
print("Uncertain extras:", ball["uncertain_extras"].sum())
print("Illegal deliveries:", (~ball["legal_ball"]).sum())

Team runs: 275375
Team wickets: 10486
Bowler wickets: 9905
Uncertain extras: 263
Illegal deliveries: 8039


In [59]:
# STEP 25: Validate delivery variables

print("DELIVERY VALIDATION")

print(
    ball[
        [
            "team_runs",
            "team_wicket",
            "bowler_wicket",
            "legal_ball",
            "uncertain_extras"
        ]
    ].isna().sum()
)

print("\nBowler wickets cannot exceed team wickets:")
print(
    (ball["bowler_wicket"] <= ball["team_wicket"]).all()
)

print("\nBowler wicket types:")
print(
    ball[ball["bowler_wicket"] == 1]
    ["dismissal_type"]
    .value_counts()
)

print("\nUncertain extras by category:")
print(
    ball[ball["uncertain_extras"]]
    ["extras_type"]
    .value_counts(dropna=False)
)

DELIVERY VALIDATION
team_runs           0
team_wicket         0
bowler_wicket       0
legal_ball          0
uncertain_extras    0
dtype: int64

Bowler wickets cannot exceed team wickets:
True

Bowler wicket types:
dismissal_type
Caught        7332
Bowled        1690
LBW            666
Stump          200
Hit Wicket      17
Name: count, dtype: int64

Uncertain extras by category:
extras_type
other      260
NaN          2
Penalty      1
Name: count, dtype: int64


In [60]:
# STEP 26: Inspect extras by category and runs

extras_check = (
    ball.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        batter_runs=("runs", "sum"),
        extras1=("extras1_runs", "sum"),
        extras2=("extras_2_runs", "sum"),
        wickets=("team_wicket", "sum")
    )
    .reset_index()
)

extras_check["team_runs"] = (
    extras_check["batter_runs"]
    + extras_check["extras1"]
    + extras_check["extras2"]
)

print(extras_check.to_string(index=False))

extras_type  deliveries  batter_runs  extras1  extras2  wickets  team_runs
   Overstep         565          834      565       48        5       1447
    Penalty           1            0        5        0        0          5
       byes         505            0      960        0        1        960
    legbyes        2661            0     3500        0        0       3500
      other         260          411      260       34        2        705
       wide        7474            0     8882        0       39       8882
        NaN      182548       259874        1        1    10439     259876


In [61]:
# STEP 27: Check available extras-related columns

print([
    col for col in ball.columns
    if "extra" in col.lower()
    or "wide" in col.lower()
    or "no_ball" in col.lower()
    or "penalty" in col.lower()
])

['extras_type', 'extras1_runs', 'extras_2_runs', 'is_wide', 'is_no_ball', 'uncertain_extras']


In [62]:
# STEP 28: Investigate extras_2_runs

extra2 = ball[ball["extras_2_runs"] > 0].copy()

print("Total records with extras_2_runs:",
      len(extra2))

print("\nBreakdown by extras category:")

print(
    extra2.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        total_extra2=("extras_2_runs", "sum"),
        batter_runs=("runs", "sum"),
        extras1=("extras1_runs", "sum")
    )
    .to_string()
)

print("\nIndividual records:")

print(
    extra2[
        [
            "match_id",
            "innings",
            "innings_ball_num",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type",
            "bowler_name"
        ]
    ].to_string(index=False)
)

Total records with extras_2_runs: 35

Breakdown by extras category:
             deliveries  total_extra2  batter_runs  extras1
extras_type                                                
Overstep             21            48            0       21
other                13            34            0       13
NaN                   1             1            0        0

Individual records:
 match_id  innings  innings_ball_num  runs extras_type  extras1_runs  extras_2_runs  is_wicket dismissal_type         bowler_name
    13523        1                 4     0    Overstep             1              4      False            NaN        Akeal Hosein
    13614        1               111     0       other             1              4      False            NaN      Mitchell Starc
    14112        1                66     0    Overstep             1              4      False            NaN  Varun Chakravarthy
    14127        2                52     0       other             1              4      Fa

In [63]:
# STEP 29: Inspect other extras records

other = ball[ball["extras_type"] == "other"].copy()

print(
    other[
        [
            "match_id",
            "innings",
            "innings_ball_num",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type",
            "bowler_name"
        ]
    ].head(30).to_string(index=False)
)

 match_id  innings  innings_ball_num  runs  extras1_runs  extras_2_runs  is_wicket dismissal_type         bowler_name
    13513        1                57     0             1              0      False            NaN         Tymal Mills
    13515        1                99     0             1              0      False            NaN       Saqib Mahmood
    13515        2               119     2             1              0      False            NaN    Sheldon Cottrell
    13614        1               111     0             1              4      False            NaN      Mitchell Starc
    13614        2                 7     0             1              0      False            NaN      Nuwan Thushara
    14094        1               119     6             1              0      False            NaN         Shivam Mavi
    14098        2               119     0             1              0      False            NaN Nathan Coulter Nile
    14101        1               105     0             1